In [9]:
import os
import json
import time
from datetime import date
from dotenv import load_dotenv
from bs4 import BeautifulSoup

# Selenium & Database Imports
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from seleniumwire import webdriver as sw_webdriver

# Import the bridge you created in database.py
from database import save_to_db, engine, text

load_dotenv(override=True)
print("Cell 1: Libraries and Database Bridge loaded.")

Cell 1: Libraries and Database Bridge loaded.


In [10]:
def get_selenium_wire_options():
    user = os.getenv("WEBSHARE_USERNAME")
    pw = os.getenv("WEBSHARE_PASSWORD")
    host = os.getenv("WEBSHARE_HOST")
    port = os.getenv("WEBSHARE_PORT")
    
    proxy_url = f"http://{user}:{pw}@{host}:{port}"
    
    return {
        'proxy': {
            'http': proxy_url,
            'https': proxy_url,
            'no_proxy': 'localhost,127.0.0.1'
        }
    }

sw_options = get_selenium_wire_options()
print(" Cell 2: Proxy options configured.")

 Cell 2: Proxy options configured.


In [11]:
print("Testing Zion_Database connection...")
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1")).fetchone()
        if result[0] == 1:
            print(" DATABASE CONNECTED SUCCESSFULLY!")
except Exception as e:
    print(f" DATABASE CONNECTION FAILED: {e}")

Testing Zion_Database connection...
 DATABASE CONNECTED SUCCESSFULLY!


In [12]:
# 1. Advanced Browser Settings
chrome_options = Options()
chrome_options.add_argument("--headless=new") # More modern headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Hide Selenium
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

today = date.today()
max_pages = 2 # Start small for testing
url_template = "https://finance.yahoo.com/research-hub/screener/sec-ind_sec-largest-equities_technology/?start={start}&count=25"

driver = sw_webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    seleniumwire_options=sw_options,
    options=chrome_options
)

try:
    for page in range(max_pages):
        start_index = page * 25
        target_url = url_template.format(start=start_index)
        print(f"📡 Scraping Page {page + 1}: {target_url}")
        
        driver.get(target_url)
        
        # --- Handle the "Consent/Cookies" wall ---
        try:
            # Look for any button that looks like "Accept" or "Agree"
            consent_btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.NAME, "agree"))
            )
            consent_btn.click()
            print("✅ Bypassed Consent Wall")
        except:
            pass # No consent wall found, proceed

        # --- Flexible Wait ---
        try:
            # Wait for ANY table row to exist, not just a specific test-id
            WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.TAG_NAME, "tr"))
            )
            time.sleep(3) # Let JS settle
        except:
            print(f"❌ Page {page+1} still failed to load. The proxy might be blocked.")
            continue

        # 3. Parse with BeautifulSoup
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Try finding rows by the attribute first, then by general table structure
        rows = soup.find_all("tr", attrs={"data-testid-row": True})
        if not rows:
            # Fallback: Find rows inside a tbody
            rows = soup.select("table tbody tr")
            
        print(f"📊 Found {len(rows)} potential stocks. Syncing...")
        
        for row in rows:
            try:
                def get_val(cell_tid):
                    cell = row.find("td", {"data-testid-cell": cell_tid})
                    return cell.get_text(strip=True) if cell else "N/A"

                ticker = get_val("ticker")
                if ticker == "N/A" or not ticker: continue
                
                stock_details = {
                    "ticker": ticker,
                    "price": get_val("intradayprice"),
                    "mkt_cap": get_val("intradaymarketcap"),
                    "volume": get_val("dayvolume")
                }
                
                # 4. SAVE TO DB
                save_to_db(
                    title=f"Yahoo Tech: {ticker}",
                    content=json.dumps(stock_details),
                    url=f"{target_url}#{ticker}",
                    report_date=today
                )
            except Exception:
                continue 

    print(f"🏁 FINISHED! Check your 'scraped_reports' table in pgAdmin.")

except Exception as e:
    print(f"❌ Scraper Error: {e}")

finally:
    driver.quit()

📡 Scraping Page 1: https://finance.yahoo.com/research-hub/screener/sec-ind_sec-largest-equities_technology/?start=0&count=25
✅ Bypassed Consent Wall
📊 Found 25 potential stocks. Syncing...
 Saved to DB: Yahoo Tech: NVDA...
 Saved to DB: Yahoo Tech: AAPL...
 Saved to DB: Yahoo Tech: MSFT...
 Saved to DB: Yahoo Tech: AVGO...
 Saved to DB: Yahoo Tech: MMU...
 Saved to DB: Yahoo Tech: ORCL...
 Saved to DB: Yahoo Tech: AMD...
 Saved to DB: Yahoo Tech: PPLTR...
 Saved to DB: Yahoo Tech: CSCO...
 Saved to DB: Yahoo Tech: AAMAT...
 Saved to DB: Yahoo Tech: LLRCX...
 Saved to DB: Yahoo Tech: INTC...
 Saved to DB: Yahoo Tech: IBM...
 Saved to DB: Yahoo Tech: KKLAC...
 Saved to DB: Yahoo Tech: TXN...
 Saved to DB: Yahoo Tech: CRM...
 Saved to DB: Yahoo Tech: AANET...
 Saved to DB: Yahoo Tech: AADI...
 Saved to DB: Yahoo Tech: AAPH...
 Saved to DB: Yahoo Tech: SHOP...
 Saved to DB: Yahoo Tech: UBER...
 Saved to DB: Yahoo Tech: QQCOM...
 Saved to DB: Yahoo Tech: PPANW...
 Saved to DB: Yahoo Tech: G